<a href="https://colab.research.google.com/github/puteriazli/sentiment-analysis-dana-application-review/blob/main/sentiment_dana_indobert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install torch transformers emoji symspellpy scikit-learn pandas

In [ ]:
import random
import re
import emoji
import torch
import numpy as np
import pandas as pd

from collections import Counter
from symspellpy import SymSpell, Verbosity
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

import torch.nn.functional as F

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LEN = 128

In [ ]:
df = pd.read_parquet("/content/drive/MyDrive/Project Mandiri/Sentiment Analysis DANA/data/ulasan_dana_cleaned.parquet")

df = df.dropna(subset=["ulasan", "rating"])
df["ulasan"] = df["ulasan"].astype(str)

def rating_to_label(r):
    if r <= 2:
        return 0
    elif r == 3:
        return 1
    return 2

df["label"] = df["rating"].apply(rating_to_label)

In [ ]:
def balance_dataset(df, n_samples_per_class=34540):
    df_neg = df[df["label"] == 0]
    df_neu = df[df["label"] == 1]
    df_pos = df[df["label"] == 2]

    print(f"Jumlah awal - Neg: {len(df_neg)}, Neu: {len(df_neu)}, Pos: {len(df_pos)}")

    df_neg = df_neg.sample(
        n=n_samples_per_class, random_state=SEED,
        replace=len(df_neg) < n_samples_per_class
    )
    df_neu = df_neu.sample(
        n=n_samples_per_class, random_state=SEED,
        replace=len(df_neu) < n_samples_per_class
    )
    df_pos = df_pos.sample(
        n=n_samples_per_class, random_state=SEED,
        replace=len(df_pos) < n_samples_per_class
    )

    df_balanced = pd.concat([df_neg, df_neu, df_pos])
    df_balanced = df_balanced.sample(frac=1, random_state=SEED).reset_index(drop=True)

    print("Distribusi baru:\n", df_balanced["label"].value_counts())
    return df_balanced

df = balance_dataset(df, n_samples_per_class=34540)

Jumlah awal - Neg: 182932, Neu: 34540, Pos: 522528
Distribusi baru:
 label
0    34540
2    34540
1    34540
Name: count, dtype: int64


In [ ]:
colloquial = pd.read_csv("/content/drive/MyDrive/Project Mandiri/Sentiment Analysis DANA/kamus/colloquial-indonesian-lexicon.csv")
slang_dict = dict(zip(colloquial["slang"], colloquial["formal"]))

def tokenize(text):
    text = re.sub(r"http\S+|www\S+", "", text.lower())
    text = re.sub(r"[^a-z ]", " ", text)
    return text.split()

counter = Counter()
for t in df["ulasan"]:
    counter.update(tokenize(t))

freq_path = "../frequency_dictionary_id.txt"
with open(freq_path, "w") as f:
    for w, c in counter.items():
        if len(w) > 2 and c >= 3:
            f.write(f"{w} {c}\n")

In [ ]:
symspell = SymSpell(max_dictionary_edit_distance=2)
symspell.load_dictionary(freq_path, 0, 1)

def normalize_elongation(w):
    return re.sub(r"(.)\1{2,}", r"\1", w)

def normalize_word(w):
    if w in slang_dict:
        return slang_dict[w]
    w = normalize_elongation(w)
    sug = symspell.lookup(w, Verbosity.CLOSEST, max_edit_distance=2)
    return sug[0].term if sug else w

def clean_text(text):
    text = emoji.demojize(text.lower(), delimiters=(" ", " "))
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z_ ]", " ", text)
    return " ".join(normalize_word(t) for t in text.split())

df["clean_text"] = df["ulasan"].apply(clean_text)

In [ ]:
POS_WORDS = {"bagus","mantap","keren","oke","baik","top","mantul","cepat","lancar"}
NEG_WORDS = {"lemot", "lambat", "lelet", "error","gagal","tidak bisa","ga bisa","crash","parah","kecewa","hapus","uninstall","lempar", "kenapa", "kok", "ngelag", "goblok", "glbk", "anjing", "ajg", "kontol", "kntl"}

def detect_sarcasm(text):
    t = text.lower()
    return any(p in t for p in POS_WORDS) and any(n in t for n in NEG_WORDS)

df["sarcasm"] = df["clean_text"].apply(detect_sarcasm).astype(int)
df["strata"] = df["label"].astype(str) + "_" + df["sarcasm"].astype(str)

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["strata"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["strata"], random_state=SEED
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.texts = df["clean_text"].tolist()
        self.labels = df["label"].tolist()

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = torch.nn.CrossEntropyLoss(reduction="none")

    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        pt = torch.exp(-ce)
        return (self.alpha[targets] * (1 - pt)**self.gamma * ce).mean()

In [ ]:
class FocalTrainer(Trainer):
    def __init__(self, focal_loss, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = focal_loss

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = self.focal_loss(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3
).to(DEVICE)

for name, param in model.base_model.named_parameters():
    if "encoder.layer" in name:
        layer = int(name.split("encoder.layer.")[1].split(".")[0])
        param.requires_grad = layer >= 8

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
class_counts = train_df["label"].value_counts().sort_index()
alpha = torch.tensor((1.0 / class_counts).values, dtype=torch.float).to(DEVICE)
alpha = alpha / alpha.sum()

focal_loss = FocalLoss(alpha)

args = TrainingArguments(
    output_dir="./model_dana_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    seed=SEED,
    report_to="none"
)

trainer = FocalTrainer(
    model=model,
    args=args,
    train_dataset=SentimentDataset(train_df),
    eval_dataset=SentimentDataset(val_df),
    focal_loss=focal_loss,
    callbacks=[EarlyStoppingCallback(3)],
    compute_metrics=lambda p: {
        "accuracy": accuracy_score(
            p.label_ids, np.argmax(p.predictions, 1)
        ),
        "macro_f1": f1_score(
            p.label_ids, np.argmax(p.predictions, 1),
            average="macro"
        )
    }
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.095041,0.094026,0.670237,0.644223


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.095041,0.094026,0.670237,0.644223
2,0.091047,0.093689,0.670623,0.657212
3,0.086275,0.093894,0.670913,0.667915
4,0.081099,0.095786,0.675835,0.666367
5,0.079029,0.096427,0.676028,0.667349


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=12955, training_loss=0.08716420034145766, metrics={'train_runtime': 2239.9154, 'train_samples_per_second': 185.043, 'train_steps_per_second': 5.784, 'total_flos': 2.726381234497536e+16, 'train_loss': 0.08716420034145766, 'epoch': 5.0})

In [ ]:
preds = trainer.predict(SentimentDataset(test_df))
y_true = test_df["label"].values
y_pred = np.argmax(preds.predictions, axis=1)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, digits=4))

[[2086 1186  182]
 [ 482 1948 1024]
 [  60  359 3035]]
              precision    recall  f1-score   support

           0     0.7938    0.6039    0.6860      3454
           1     0.5577    0.5640    0.5608      3454
           2     0.7156    0.8787    0.7888      3454

    accuracy                         0.6822     10362
   macro avg     0.6890    0.6822    0.6785     10362
weighted avg     0.6890    0.6822    0.6785     10362



In [ ]:
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
#print("Cohen Kappa:", cohen_kappa_score(y_true, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, digits=4))

Accuracy: 0.6822042076819147
Macro F1: 0.6785333656693976

Classification Report:

              precision    recall  f1-score   support

           0     0.7938    0.6039    0.6860      3454
           1     0.5577    0.5640    0.5608      3454
           2     0.7156    0.8787    0.7888      3454

    accuracy                         0.6822     10362
   macro avg     0.6890    0.6822    0.6785     10362
weighted avg     0.6890    0.6822    0.6785     10362



In [ ]:
label_map = {0:"negative",1:"neutral",2:"positive"}

def predict_sentiment(text):
    clean = clean_text(text)
    enc = tokenizer(clean, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(DEVICE)

    with torch.no_grad():
        logits = model(**enc).logits
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]

    pred = probs.argmax()

    if pred == 2 and detect_sarcasm(clean):
        return {"label":"negative","reason":"sarcasm","confidence":float(probs[pred])}

    if probs.max() < 0.45:
        return {"label":"neutral","reason":"low_confidence","confidence":float(probs.max())}

    return {"label":label_map[pred],"confidence":float(probs[pred])}

In [ ]:
# contoh
predict_sentiment("lumayan")

{'label': 'neutral', 'confidence': 0.5670137405395508}